# Bedrock Module · Day 12 — Knowledge Bases (RAG)

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~2.5 hrs

Yesterday you ran a model. But a raw model doesn't know **Barclays' own documents** — its policies, FAQs, product terms. **Retrieval-Augmented Generation (RAG)** fixes that: fetch the *relevant* snippets from your documents, then ask the model to answer **using them**. A **Bedrock Knowledge Base** is AWS's *managed* RAG: point it at an S3 prefix and it handles **chunk → embed → store → retrieve → generate** for you. Today you build that pipeline by hand (so you understand every stage) and drive it through the two APIs you'll use for real: **`retrieve`** and **`retrieve_and_generate`**.

> **Keyless & offline.** No AWS account, no OpenSearch. We build a tiny real vector index over canned banking docs (deterministic embeddings + cosine) behind a **`FakeBedrockAgentRuntime`** with the same method names and response shapes as the real client. Promote to production by swapping the client and pointing at your S3 prefix — the request/response shapes are identical.


## 0. Setup — the corpus + a keyless client

These docs stand in for what would sit under `s3://barclays-kb-docs/policies/` (from this morning's S3 lesson). The `FakeBedrockAgentRuntime` will chunk, embed, and index them, then serve `retrieve` and `retrieve_and_generate`.


In [1]:
import re, math

DOCS = {
    "policies/aml.txt":  ("Anti-Money-Laundering policy. Any cash transaction over 10000 GBP must be "
                          "reported to the compliance team within 24 hours. Structuring transactions "
                          "to stay under the limit is itself a reportable red flag."),
    "policies/kyc.txt":  ("Know Your Customer policy. Verify a customer's identity with two documents "
                          "before onboarding. Refresh KYC every three years, or sooner for high-risk "
                          "accounts."),
    "policies/cards.txt":("Lost or stolen card procedure. Customers freeze the card instantly in the "
                          "mobile app. A replacement card is issued and arrives within three business "
                          "days. No liability for fraud reported within 24 hours."),
    "policies/fees.txt": ("International transfer fees. SWIFT payments cost a flat 15 GBP plus an "
                          "exchange-rate margin. Transfers inside the EU under 1000 GBP are free."),
}
print(len(DOCS), "source documents")


4 source documents


## 1. Stage 1 — **chunk** the documents

Models retrieve *passages*, not whole files, so each doc is split into **chunks**. Chunk size is the single biggest RAG tuning knob: **too big** = the answer is diluted by irrelevant text and you waste tokens; **too small** = a fact gets split across chunks and retrieval misses it. We'll chunk by sentence count so we can compare sizes in §6.


In [2]:
def chunk(text, sentences_per_chunk=1):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    out=[]
    for i in range(0, len(sents), sentences_per_chunk):
        out.append(" ".join(sents[i:i+sentences_per_chunk]))
    return out

def build_chunks(docs, size=1):
    rows=[]
    for key, text in docs.items():
        for j, c in enumerate(chunk(text, size)):
            rows.append({"source": key, "chunk_id": f"{key}#{j}", "text": c})
    return rows

chunks = build_chunks(DOCS, size=1)
print(f"{len(DOCS)} docs -> {len(chunks)} chunks")
for r in chunks[:3]: print(" ", r["chunk_id"], "::", r["text"][:60], "...")


4 docs -> 13 chunks
  policies/aml.txt#0 :: Anti-Money-Laundering policy. ...
  policies/aml.txt#1 :: Any cash transaction over 10000 GBP must be reported to the  ...
  policies/aml.txt#2 :: Structuring transactions to stay under the limit is itself a ...


☝ Each chunk keeps a pointer back to its **source** — that pointer is what becomes a **citation** later. A managed Knowledge Base does this step for you (fixed-size, semantic, or hierarchical chunking); doing it by hand shows why the size choice matters.


## 2. Stage 2 — **embed** each chunk into a vector

An **embedding** turns text into a list of numbers so that *similar meaning → nearby vectors*. In production this is a call to `amazon.titan-embed-text-v2` (via `invoke_model`, the per-model API from Day 11). Here we use a small deterministic bag-of-words vector so the notebook stays offline — the *shape of the pipeline* is identical.


In [3]:
# drop common stop-words so overlap reflects *meaning*, not filler like 'the'/'is'
STOP = set("a an the is are was were be to of in on for and or if it its do does my your "
           "i you what how when where which that this these those with within under over as at "
           "any must not no can will shall from into by up out sooner itself".split())
def tokens(text): return [w for w in re.findall(r'[a-z]+', text.lower()) if w not in STOP]

VOCAB = sorted(set(w for r in chunks for w in tokens(r['text'])))
IDX = {w:i for i,w in enumerate(VOCAB)}

def embed(text):
    v=[0.0]*len(VOCAB)
    for w in tokens(text):
        if w in IDX: v[IDX[w]] += 1.0
    return v

def cosine(a,b):
    dot=sum(x*y for x,y in zip(a,b))
    na=math.sqrt(sum(x*x for x in a)); nb=math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

for r in chunks: r["vec"]=embed(r["text"])
print(f"embedded {len(chunks)} chunks into {len(VOCAB)}-dim vectors")


embedded 13 chunks into 66-dim vectors


☝ Real embeddings are ~1024 dense dimensions from a trained model; ours are sparse word counts. Both obey the same rule the retriever relies on: **cosine similarity ranks by meaning**. In AWS these vectors land in a **vector store** — OpenSearch Serverless or Aurora **pgvector** — which is just a database that indexes vectors for fast nearest-neighbour search.


## 3. Stage 3 — the vector store + a keyless client

Now wrap it in a client with the real Bedrock Agent Runtime method names. `retrieve` returns ranked chunks with **scores + source location**; `retrieve_and_generate` does that **and** asks a model to write the answer with citations.


In [4]:
class FakeBedrockAgentRuntime:
    def __init__(self, index): self.index=index      # list of chunk dicts with 'vec'
    def retrieve(self, query, top_k=3):
        q=embed(query)
        scored=sorted(self.index, key=lambda r: cosine(q, r["vec"]), reverse=True)
        return {"retrievalResults": [
            {"content": {"text": r["text"]},
             "location": {"s3Location": {"uri": f"s3://barclays-kb-docs/{r['source']}"}},
             "score": round(cosine(q, r["vec"]), 3)}
            for r in scored[:top_k] if cosine(q, r["vec"]) > 0]}
    def retrieve_and_generate(self, query, top_k=3):
        got=self.retrieve(query, top_k)["retrievalResults"]
        if not got:
            return {"output":{"text":"I don't have a document covering that."},"citations":[]}
        # 'generation' = grounded answer: stitch the top passages (a real model would paraphrase)
        answer=" ".join(g["content"]["text"] for g in got[:2])
        cites=[{"retrievedReference": g["location"], "text": g["content"]["text"]} for g in got]
        return {"output":{"text": answer}, "citations": cites}

kb = FakeBedrockAgentRuntime(chunks)
print("knowledge base ready:", len(chunks), "chunks indexed")


knowledge base ready: 13 chunks indexed


## 4. `retrieve` — fetch the relevant passages (no model yet)

Use `retrieve` when *your own* code will do something with the passages (feed a custom prompt, show sources, apply filters). It returns ranked chunks with a **score** and the **S3 location** each came from.


In [5]:
res = kb.retrieve("How quickly must a large cash transaction be reported?", top_k=3)
for hit in res["retrievalResults"]:
    print(f"score {hit['score']:.3f} · {hit['location']['s3Location']['uri']}")
    print(f"   {hit['content']['text'][:90]}...")


score 0.655 · s3://barclays-kb-docs/policies/aml.txt
   Any cash transaction over 10000 GBP must be reported to the compliance team within 24 hour...
score 0.289 · s3://barclays-kb-docs/policies/cards.txt
   No liability for fraud reported within 24 hours....


☝ The AML chunk ranks top because its words overlap the question's meaning — that's cosine doing its job. The **score** lets you set a floor (drop weak hits) and the **s3Location** is the breadcrumb back to the source file — the raw material of a citation.


## 5. `retrieve_and_generate` — the full RAG answer, with citations

This is the one-call managed RAG: retrieve **and** generate. It returns `output.text` (the grounded answer) plus a `citations` list linking each claim back to its source. Because the answer is built *only* from retrieved text, the model can't invent a policy that isn't in your documents — the whole point in a regulated bank.


In [6]:
def ask_kb(question):
    r = kb.retrieve_and_generate(question)
    print("Q:", question)
    print("A:", r["output"]["text"])
    print("sources:")
    for c in r["citations"]:
        print("   -", c["retrievedReference"]["s3Location"]["uri"])
    print()

ask_kb("What do I do if my card is stolen?")
ask_kb("Is an EU transfer of 500 GBP free?")
ask_kb("What is the capital of France?")   # not in the docs -> honest 'I don't know'


Q: What do I do if my card is stolen?
A: Lost or stolen card procedure. Customers freeze the card instantly in the mobile app.
sources:
   - s3://barclays-kb-docs/policies/cards.txt
   - s3://barclays-kb-docs/policies/cards.txt
   - s3://barclays-kb-docs/policies/cards.txt

Q: Is an EU transfer of 500 GBP free?
A: Transfers inside the EU under 1000 GBP are free. International transfer fees.
sources:
   - s3://barclays-kb-docs/policies/fees.txt
   - s3://barclays-kb-docs/policies/fees.txt
   - s3://barclays-kb-docs/policies/aml.txt

Q: What is the capital of France?
A: I don't have a document covering that.
sources:



☝ Two grounded answers with sources, and — crucially — the France question returns **‘I don't have a document covering that’** instead of guessing. That refusal is a *feature*: a RAG bot that only speaks from approved documents is auditable. Every answer carries its S3 citations, so a reviewer can open the exact source.


## 6. Tuning — **chunk size** changes citation quality

The plan's task: *compare citation quality across chunk sizes*. Same corpus, same question, different `sentences_per_chunk`. Watch how the top chunk's **precision** changes — bigger chunks pull in extra, less-relevant sentences (lower score, noisier citation); smaller chunks are tighter but risk splitting a fact.


In [7]:
question = "How long until a replacement card arrives?"

for size in (1, 2, 3):
    idx = build_chunks(DOCS, size=size)
    for r in idx: r["vec"]=embed(r["text"])
    top = FakeBedrockAgentRuntime(idx).retrieve(question, top_k=1)["retrievalResults"][0]
    words = len(top["content"]["text"].split())
    print(f"chunk={size} sent | score {top['score']:.3f} | {words:>2} words | {top['content']['text'][:70]}...")


chunk=1 sent | score 0.655 | 11 words | A replacement card is issued and arrives within three business days....
chunk=2 sent | score 0.522 | 19 words | A replacement card is issued and arrives within three business days. N...
chunk=3 sent | score 0.602 | 25 words | Lost or stolen card procedure. Customers freeze the card instantly in ...


☝ Smaller chunks score **higher** here (the answer sentence isn't diluted) but a too-small chunk can strand context a neighbour sentence held. The real trade-off: pick the smallest chunk that still contains a *complete* fact. This is exactly the dial you'd tune in a Bedrock KB (fixed-size vs semantic chunking) and measure with a retrieval-eval set (Day 24).


## 7. How the managed service maps to what you built

| You wrote | Bedrock Knowledge Base does it via |
|---|---|
| `DOCS` dict | **S3 data source** (a bucket prefix — this morning's S3 lesson) |
| `chunk()` | ingestion **chunking strategy** (fixed / semantic / hierarchical) |
| `embed()` | **Titan / Cohere** embedding model (`invoke_model`) |
| `cosine` index | **vector store**: OpenSearch Serverless or Aurora **pgvector** |
| `retrieve()` | **`Retrieve` API** |
| `retrieve_and_generate()` | **`RetrieveAndGenerate` API** |

To go live: create a KB in the Bedrock console, point it at `s3://barclays-kb-docs/policies/`, pick an embedding model + vector store, click **Sync**. Then:

```python
import boto3
agent = boto3.client("bedrock-agent-runtime", region_name="eu-west-2")
agent.retrieve_and_generate(
    input={"text": question},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {"knowledgeBaseId": "KB123",
            "modelArn": "eu.anthropic.claude-sonnet-4-5-20250929-v1:0"}})
```


## 8. Recap — you built RAG

| Stage | What it does | Real Bedrock piece |
|---|---|---|
| **source** | documents in S3 | S3 data source |
| **chunk** | split into passages | chunking strategy |
| **embed** | text → vector | Titan Embeddings |
| **store** | index vectors | OpenSearch / pgvector |
| **retrieve** | rank by cosine | `Retrieve` API |
| **generate** | grounded answer + citations | `RetrieveAndGenerate` API |

**Barclays lens:** a Knowledge Base is *managed, cited, auditable* RAG — the model answers only from approved documents and every answer links to its source. That citation trail is what makes it usable for compliance and customer-facing tools.